# Using `HDFGClean` with `hdsims`

This notebook will show you how to run the foreground (FG) cleaning using the `run_hdfgclean.py` python script provided in the `hdfgclean` repository. We will assume that you have read the readme files of both `hdfgclean` and `hdsims`. For additional information not found below, you may also refer to the documentation (the docstrings under function/method/class/etc. definitions) or the examples provided with `hdfgclean`, and you may refer to MacInnis et. al. (2026) (**TODO:LINK2PAPER**) for a full description of the FG-cleaning procedure.

In the next cell, we import the packages and modules that will be used below:

In [ ]:
import os
from hdfgclean import hdfgclean, hdfgclean_utils

---

# Configuration file used to initialize `HDFGClean`

Here, we will save the YAML configuration file that can be used to initialize `HDFGClean` and run the FG cleaning. We describe the required and optional arguments that can be passed to `HDFGClean` in the "Required arguments" and "Optional keyword arguments" sections. In the "Save the configuration file" section, you will define the required arguments, add any additional keyword arguments you would like to use, and save the configuration file.

---

## Required arguments

When initializing `HDFGClean`, you **must** always provide the (absolute) paths to:
- your `hd_sims_dir` where the simulations will be saved (see [hdsims](https://github.com/CMB-HD/hdsims) for more information)
- an `output_dir` where the output from FG cleaning will be saved

---

## Optional keyword arguments

As mention in the `hdfgclean` [readme](https://github.com/CMB-HD/hdfgclean/blob/main/README.md#using-hdfgclean-with-hdsims) file, the `HDFGClean` class accepts all keyword arguments that may be passed to the `HDSims` class (in the `hdsims.py` module of the `hdsims` package) and to the `FGClean` class (in the `fgclean.py` module of the `hdfgclean` package).

Below, we highlight some of the keyword arguments accepted by `HDFGClean`, which are passed to either `HDSims` or `FGClean`. All optional keyword arguments and their default values are given in the `hdfgclean_defaults.yaml` file provided in the `hdfgclean` repository; these are the values used in MacInnis et. al. (2026).

### For the simulations (passed to `HDSims`)

**Reminder**: you should always generate the set of simulations to be FG cleaned (and the set of maps used to quantify the noise in the matched filter) using `hdsims` *before* initializing the `HDFGClean` class (which is done by `run_hdfgclean.py`); see the [readme](https://github.com/CMB-HD/hdfgclean/blob/main/README.md#before-using-hdfgclean) for more information.

`HDFGClean` accepts all keyword arguments that can be passed to `HDSims`; however, `pol` will be always be set to `False`, since polarization maps are not needed to run the FG cleaning. Note that you may still generate the simulations with `pol=True` (which is the `HDSims` default) and then use them with `HDFGClean`.

These arguments are described in detail in the `hdsims` package. Below, we summarize a few of the settings you may wish to change:

- You may change the location of your maps by passing a different `ra_ctr` and `dec_ctr` (both in degrees): these are the right ascension (R.A.) and declination (dec.) coordinates, respectively, of the center of the maps. The defaults are `ra_ctr=6` and `dec_ctr=6`.
- You may change size of your maps by passing a different `width` and `height` (both in degrees). The defaults are `width=10` and `height=10`.
- You may change the width of the region at the map edges used to apodize them by passing a different `apod_width` (in degrees). The default is `apod_width=0.5`.

The maps being FG cleaned must be apodized (which is done automatically by `HDFGClean`), so they will initially be extended by the `apod_width` on each side; this leaves the inner `width` $\times$ `height` region un-apodized.

We do *not* recommend changing which map frequencies are used: the 90 and 148 GHz maps have the lowest noise, and the maps at higher frequencies (219 GHz and 277 GHz) are needed to FG-clean the 90 and 148 GHz maps down to the levels reported in MacInnis et. al. (2026)

### For FG cleaning (passed to `FGClean`)

Many of the optional keyword arguments accepted by the `FGClean` class are automatically set by the `HDFGClean` class. Below, we describe some of the other arguments that you may wish to change.

#### Dividing the maps into smaller patches

As mentioned in the `hdfgclean` [readme](https://github.com/CMB-HD/hdfgclean/blob/main/README.md#using-mpi-strongly-recommended), we divide maps that are larger than some maximum size ($3^\circ \times 3^\circ$ by default) into a grid of smaller patches, and run the full FG cleaning procedure on each individual patch. The patches themselves are apodized (by default using an apodization width of 0.25 degrees). 
- The maximum patch size includes the extra area used to apodize the input maps; e.g., a set of simulations generated with `width=3`, `height=3`, and `apod_width=0.25` would exceed the default maximum of $3^\circ \times 3^\circ$, since the apodized maps are actually $3.5^\circ \times 3.5^\circ$.

You may change the maximum width and height of each smaller patch by passing a different `max_patch_size` (in degrees), and change the apodization width of the patches by passing a different `patch_apod_width` (in degrees). The defaults are `max_patch_size=3` and `patch_apod_width=0.25`.



Note that we use a second set of simulations, generated with `hdsims`, to quantify the noise in the matched filter calculations (see the `hdfgclean` [readme](https://github.com/CMB-HD/hdfgclean/blob/main/README.md#before-using-hdfgclean) and the relevant section below); the `max_patch_size` should not exceed the size of these maps. 
- By default, we use maps that were generated with `width=3`, `height=3`, and `apod_width=0.25` (along with different random seeds and a different `ra_ctr`).
- If you generate a new set of simulations to use in the matched filter calculations, then you must make sure that the `max_patch_size` does not exceed the `width` or `height` (whichever is smaller) you used to generate them.

#### Iterative FG cleaning

By default, the FG cleaning will detect and remove CIB+radio point sources and tSZ clusters from the maps. Pass `subtract_sources=False` if you would like to completely turn off the source-subtraction part, or pass `subtract_clusters=False` if you would like to completely turn off the cluster-subtraction part. (Note that at least one of these must be `True`, or else there is nothing for the code to do!)
- If you are using simulations without CIB and radio sources, `subtract_sources` will always be set to `False`; or, if your simulations do not contain the tSZ, `subtract_clusters` will always be set to `False`.

As summarized in the `hdfgclean` [readme](https://github.com/CMB-HD/hdfgclean/blob/main/README.md#overview-of-foreground-cleaning) file (and described in detail in MacInnis et. al. 2026), on each FG cleaning iteration, we apply a matched filter to the map(s) to isolate point sources or clusters. The resulting filtered map contains the signal we are trying to measure (and remove). We construct an "SNR map" with the signal-to-noise ratio (SNR) of each pixel in the filtered map: the filtered map itself is the "signal" part, and we refer to the "noise" part as the "RMS map". This RMS map is constructed by diving the filtered map into a grid; the value of the RMS map within a given grid cell is the standard deviation of the pixels in the filtered map within that grid cell.


- You may change the default set of SNR thresholds used to detect point sources or clusters by passing a different list to `sources_snr_threshold_list` or `clusters_snr_threshold_list`, respectively. 
  - The defaults are `sources_snr_threshold_list = [250, 100, 75, 50, 40, 30, 25, 20, 15, 12.5, 10, 7.5, 5, 4]` and `clusters_snr_threshold_list = [50, 25, 15, 12.5, 10, 7.5, 5, 4]`.
- You may change the RMS map grid size (width and height of grid cells) used when filtering the maps for point sources or clusters by passing a different `rms_gw_sources` or `rms_gw_clusters`, respectively (both in arcminutes).
  - The defaults are `rms_gw_sources=10` and `rms_gw_clusters=40`.

#### tSZ cluster profiles

We use a set of radial cluster profiles when filtering the maps to isolate tSZ clusters. Each detected cluster is subtracted from the maps using the profile that gave the highest SNR detection for that cluster.

The default cluster profiles are a set of 11 Gaussians with standard deviations of 0.25', 0.3', 0.35', 0.4', 0.45', 0.5', 0.55', 0.6', 0.65', 0.7', and 0.75', each with zero mean and a maximum amplitude of one.

You may use a different set of cluster profiles by passing a dictionary of `cluster_profiles` to `HDFGClean` (see below). In this case, you *must* also pass a short description (`str`) of the cluster profiles to `cluster_profiles_info`; it is used in the output file names, so it should not contain any special characters. This allows you to run the FG cleaning (on the same set of maps and with the same `output_dir`) multiple times, each time using a different set of `cluster_profiles` and `cluster_profiles_info`, without having to run the source-subtraction again.


Each item in the `cluster_profiles` dictionary corresponds to a single radial profile function. The key (`str`) should be a short, unique label for the profile. The corresponding value should also be a dictionary, with the following keys and values:
- `'radial_prof_func'` : A callable radial profile function. The first argument must be the angular distance (in arcminutes) from the origin (cluster center).
- `'rmax'` : A maximum angular distance (in arcminutes) from the origin, beyond which the radial profile is truncated.
- `'args'` : A list of any additional positional arguments to pass to the profile function.
- `'kwargs'` : A dictionary of any additional keyword arguments to pass to the profile function.

For example, suppose you want to use only two Gaussian profiles: one with $\sigma=0.5'$, and one with $\sigma=1'$. Below, we show how to define the `cluster_profiles` dictionary and the `cluster_profiles_info` that you would pass to `HDFGClean`:

```python
import numpy as np

def gaussian(x, sigma, mu=0, norm=None):
    """Gaussian distribution with standard deviation  `sigma` and mean 
    `mu` (default is `0`), normalized to a maximum amplitude of `norm` 
    (default is `1 / sqrt(2 * pi * sigma**2)`)
    """
    if norm is None:
        norm = 1 / (sigma * np.sqrt(2 * np.pi))
    return norm * exp(-0.5 * (x - mu)**2 / sigma**2)


cluster_profiles_info = 'my2profiles'
cluster_profiles = {}
for sigma in [0.5, 1]:
    profile_name = f'{sigma}arcmin'
    # maximum distance from origin to use:
    profile_rmax = 5 * sigma 
    # arguments passed to the `gaussian` function:
    profile_args = [sigma] 
    # keyword arguments passed to the `gaussian` function:
    profile_kwargs = {'norm': 1}
    # add this profile to the dictionary:
    cluster_profiles[profile_name] = {'radial_prof_func': gaussian, 'rmax': profile_rmax,
                                      'args': profile_args, 'kwargs': profile_kwargs}
```

**Note**: if you pass a `cluster_profiles` dictionary to the `HDFGClean.save_config` method, you must pass `safe=False` to the `HDFGClean.from_config` method when using your configuration file; this allows the execution of *any* arbitrary python code saved in the YAML file (by passing `Loader=yaml.Loader` when calling `yaml.load`). Therefore, you should *only* use this option with YAML files from a trusted source (i.e., yourself); see the [PyYAML documentation](https://pyyaml.org/wiki/PyYAMLDocumentation) for more information. Alternatively, if you want to only use `yaml.safe_load`, you can pass the `cluster_profiles` dictionary as a keyword argument to the `from_config` method, instead of saving the dictionary in your configuration file.

#### Maps used to quantify the noise in the matched filters

By default, we use a set of $3^\circ \times 3^\circ$ simulations generated by the `hdsims` package, along with pre-computed catalogs of _detected_ (using `hdfgclean`) point sources and clusters in those maps (see below), to quantify the noise when calculating the matched filters; this is what determines the default maximum patch size. These default maps are located on a patch of sky that does not overlap with the set of $10^\circ \times 10^\circ$ simulations that are FG-cleaned by default, and they have different realizations of the CMB and instrumental noise. See the `hdfgclean` [readme](https://github.com/CMB-HD/hdfgclean/blob/main/README.md#using-hdfgclean-with-hdsims) for instructions on how to generate the default set of maps used for this purpose, which *must* be done before running the FG cleaning. 


If you would like to use a different set of simulations (generated by `hdsims`) for this purpose, e.g. if you would like to increase the maximum patch size, we provide instructions to do so below. For some context, we also provide some additional details about the matched filter calculation, which you are not required to read. We will refer to the set of simulations used to quantify the noise in the matched filter calculations as the "noise maps", to clearly distinguish them from the set of simulations being FG-cleaned.

We apply a matched filter to a map to isolate a given signal (CIB and radio point sources or tSZ clusters); this is necessary because the map also contains other signals (e.g., the CMB itself) and instrumental noise. 

In general, a map $T$ ("T" for CMB temperature map) can be written as the sum of the signal we want to isolate, and everything else in the map: $T = T_\mathrm{signal} + T_\mathrm{other}$. The matched filter calculation requires an estimate of the power spectrum, $P_\mathrm{other}\left(\vec{k}\right)$, of $T_\mathrm{other}$. We assume that we only know $T$ (if we had the true $T_\mathrm{other}$ map, then we could just subtract it from $T$ to get $T_\mathrm{signal}$), and use the "noise maps" described above to make an estimate $\hat{T}_\mathrm{other}$ of the true $T_\mathrm{other}$ map.

Note that the components in $T_\mathrm{other}$ change depending on the FG-cleaning step:
- When we filter the maps to isolate CIB and radio point sources, $T_\mathrm{other}$ contains (by default) the lensed CMB, kSZ, tSZ, and instrumental noise.
- When we filter the maps to isolate tSZ clusters _after_ subtracting all detected point sources, $T_\mathrm{other}$ contains (by default) the lensed CMB, kSZ, _residual_ CIB and radio sources (i.e., the dim sources that were not detected), and instrumental noise.
- After subtracting all detected clusters, we again filter the maps to identify any remaining $|\mathrm{SNR}| \geq 5$ point sources (or clusters); at this point, the $T_\mathrm{other}$ map for the point source filter contains the lensed CMB, kSZ, _residual_ tSZ, and instrumental noise.

Therefore, in addition to the noise maps themselves, we also need maps of the _detected_ point sources and clusters in these maps (to make maps of the residual point sources or clusters). These are obtained by running the FG-cleaning procedure (using `HDFGClean.run_hdfgclean`) on the noise maps.

**How to generate new "noise maps"**:

__Step 1__: Generate the simulations to use as your noise maps with `hdsims`
- Initialize the `HDSims` class (in the `hdsims.py` module of the `hdsims` package) by passing keyword arguments for the noise maps (size, location, random noise seeds, etc.).
  - Note that you must change the random CMB and noise seeds to generate different realizations of the unlensed CMB and instrumental noise, respectively.
  - You may set `pol=False` if you do not want to also generate CMB polarization maps (Q and U), which are not used in the matched filter calculations, and you may change the default set of frequencies to generate only the maps that you need.

See the `hdsims` github repository for detailed instructions on how to generate a new set of simulations. Here is a brief example to generate a set of $5^\circ \times 5^\circ$ noise maps; we put the keyword arguments in a dictionary named `hdsims_kwargs`:

```python
from hdsims import hdsims

hd_sims_dir = '/path/to/sims' 
lowres_sims_dir = '/path/to/lowres_sims'
freqs = [90, 148, 219, 277] # default list also includes 30 and 350 GHz

ra_ctr = 60        # default is 6; can keep the default `dec_ctr=6`
width = 5          # default is 10
height = 5         # default is 10
cmb_seed = 92      # default is 58
noise_seeds = {freq: 2*freq for freq in freqs} # defaults are 2,3,4,5 for these `freqs`
hdsims_kwargs = {'ra_ctr': ra_ctr, 'width': width, 'height': height,
                 'cmb_seed': cmb_seed, 'noise_seeds': noise_seeds}

simlib = hdsims.HDSims(hd_sims_dir, lowres_sims_dir=lowres_sims_dir,
                       freqs=freqs, pol=False, **hdsims_kwargs)
simlib.generate_hd_sims()
```




The default set of noise maps were generated with `ra_ctr=26`, `width=3`, `height=3`, `apod_width=0.25`, `cmb_seed=15`, and `noise_seeds={90: 90, 148: 148, 219: 219, 277: 277}`.




__Step 2__: Run the FG cleaning on these maps
- Pass the same set of keyword arguments for these noise maps to `HDFGClean`, and then run the FG cleaning on them to save catalogs of detected point sources and clusters

This can be done using the `run_hdfgclean.py` python script we provide, but the example below shows how to do it in your own python script (see the [readme](https://github.com/CMB-HD/hdfgclean/blob/main/README.md#using-mpi-strongly-recommended) before running the FG cleaning):

```python
from hdfgclean import hdfgclean

hd_sims_dir = '/path/to/sims' 
output_dir = '/path/to/output'

ra_ctr = 60        
width = 5          
height = 5         
cmb_seed = 92      
noise_seeds = {freq: 2*freq for freq in freqs} 
hdsims_kwargs = {'ra_ctr': ra_ctr, 'width': width, 'height': height,
                 'cmb_seed': cmb_seed, 'noise_seeds': noise_seeds}

hdfgcleanlib = hdfgclean.HDFGClean(output_dir, hd_sims_dir, 
                                   # pass any other keyword arguments here
                                   **hdsims_kwargs)
hdfgcleanlib.run_hdfgclean(make_masks=False, take_power=False, 
                           match_to_true_catalogs=False, make_plots=False)
```

**How to initialize `HDFGClean` to use these "noise maps"**:

The information about the noise maps is passed to `HDFGClean` via the `noise_maps_kwargs` keyword argument. This is a dictionary that must contain the keyword arguments for the noise maps (the `hdsims_kwargs` defined above), and paths to the catalogs of detected point sources and clusters in these maps. You may also pass a new `max_patch_size` (which cannot exceed the size of the noise maps) and/or `patch_apod_width`



For example, assuming the two code blocks above were run:

- Get the paths to the necessary catalogs:

```python
from hdfgclean import hdfgclean
hd_sims_dir = '/path/to/sims' 
output_dir = '/path/to/output'
noise_maps_hdfgcleanlib = hdfgclean.HDFGClean(output_dir, hd_sims_dir, 
                                              # pass any other keyword arguments here
                                              **hdsims_kwargs)
noise_map_clusters_catalog = noise_maps_hdfgcleanlib.catalog_of_subtracted_clusters_fname()
noise_map_sources_catalogs = {}
freqs = noise_maps_hdfgcleanlib.freqs
for freq in freqs:
    catalog_file = noise_maps_hdfgcleanlib.catalog_of_subtracted_sources_fname(freq)
    noise_map_sources_catalogs[freq] = catalog_file
```

- Define the keyword arguments that were passed to `HDSims` to generate the noise maps:

```python
noise_maps_ra_ctr = 60        
noise_maps_width = 5          
noise_maps_height = 5         
noise_maps_cmb_seed = 92      
noise_maps_noise_seeds = {freq: 2*freq for freq in freqs} 
```

- Define the `noise_maps_kwargs` dictionary:

```python
noise_maps_kwargs = {'ra_ctr': noise_maps_ra_ctr, 
                     'width': noise_maps_width, 
                     'height': noise_maps_height,
                     'cmb_seed': noise_maps_cmb_seed, 
                     'noise_seeds': noise_maps_noise_seeds,
                     'sources_to_subtract_catalog_files': noise_map_sources_catalogs,
                     'clusters_to_subtract_catalog_file': noise_map_clusters_catalog}
```

- Then, either pass it directly to `HDFGClean`:

```python
hdfgcleanlib = hdfgclean.HDFGClean(output_dir, hd_sims_dir, 
                                   # pass any other keyword arguments here
                                   noise_maps_kwargs=noise_maps_kwargs)
```

- Or, save a configuration `.yaml` file that includes the `noise_maps_kwargs`, and pass that to `HDFGClean`
  - This way, you only need to define your `hd_sims_dir`, `output_dir`, and `config_file` to initialize `HDFGClean` in the future

```python
config_file = '/path/to/my_hdfgclean_config.yaml'
hdfgclean.HDFGClean.save_config(config_file, hd_sims_dir, output_dir=output_dir, 
                                # pass any other keyword arguments here
                                noise_maps_kwargs=noise_maps_kwargs)
hdfgcleanlib = hdfgclean.HDFGClean.from_config(config_file)
```

---

## Save the configuration file

In the cell below, we will save the configuration file used to run the FG cleaning.

You **must** provide the two required arguments: your `hd_sims_dir` and an `output_dir`. We recommend that you place both of these directories outside of the `hdfgclean` repository. The amount of storage required will depend on the size of your simulations; for reference,
- For 100-square-degree maps, the simulations require about 70 GB, and the FG cleaning produces about 45 GB of files.
- For four-square-degree maps, the simulations require about 7 GB, and the FG cleaning produces about 2 GB of files.

You may use a different configuration file name by changing the `config_fname`. By default, it will be a file named `run_hdfgclean.yaml` saved in your `output_dir`. To overwrite an existing file, set `overwrite_config = True`.

You may also pass any additional keyword arguments for `HDFGClean` to the `save_config` method used to save the configuration file. For example (from the `examples/hdfgclean_example_2x2.ipynb` notebook), after defining the `hd_sims_dir`, `output_dir`, and `config_file`:

```python
width = 2 # width (in degrees) of the sims being FG cleaned
height = width # height of sims
apod_width = 0.2 # apod. width (in degrees) to apodize sims
noise_maps_kwargs = {'width': width, 'height': height, 'apod_width': apod_width}
patch_apod_width = apod_width
hdfgclean.HDFGClean.save_config(config_file, hd_sims_dir, output_dir=output_dir,
                                width=width, height=height, apod_width=apod_width,
                                patch_apod_width=patch_apod_width, 
                                noise_maps_kwargs=noise_maps_kwargs)
```

In [ ]:
# you must define the `hd_sims_dir` and `output_dir`:
hd_sims_dir = 
output_dir =

# you may change the `config_file`:
config_file = os.path.join(output_dir, 'run_hdfgclean.yaml')
overwrite_config = False

# save the configuration file; this is where you may pass any additional keyword arguments:
hdfgclean.HDFGClean.save_config(config_file, hd_sims_dir, output_dir=output_dir,
                                # pass any other keyword arguments here
                                overwrite=overwrite_config)

---

# Running the FG cleaning with `run_hdfgclean.py`

Here, we will print out the command(s) to run all FG cleaning steps using your configuration file and the `run_hdfgclean.py` python script provided in the `hdfgclean` repository. 

**Please refer to the `hdfgclean` [readme](https://github.com/CMB-HD/hdfgclean/blob/main/README.md) file before running these commands**, especially the "[Using MPI](https://github.com/CMB-HD/hdfgclean/blob/main/README.md#using-mpi-strongly-recommended)" and "[Before using `HDFGClean`](https://github.com/CMB-HD/hdfgclean/blob/main/README.md#before-using-hdfgclean)" sections.

The commands you will need to run (and the order in which to run them, if applicable) will be printed out by the `print_run_hdfgclean_commands` function below. You may change any of the keyword arguments passed to that function; here, they are set to their default values. These options are:

- `safe_load` (`bool`): By default, we use `yaml.safe_load` to load the YAML files. As the name implies, this is "safe" because it only loads in the content in a YAML file without executing any code. If you have saved, e.g., numpy arrays in your YAML file, it cannot be loaded by `safe_load`; see the [PyYAML documentation](https://pyyaml.org/wiki/PyYAMLDocumentation) for more information. In this case, you have two options: either (1) make sure only pure python objects are written to the file (e.g., converting arrays to lists), or (2) pass `safe_load=False`, which will load the file by calling `yaml.load` and passing `Loader=yaml.Loader`.
- `save_maps` (`bool`): Whether to save the FG-cleaned maps, and maps of all detected point sources and clusters (when applicable), at each map frequency.
- `masks` (`bool`): Whether to make the masks after FG cleaning.
- `mask_freqs` (`list` of `int`): Make the mask for these map frequencies. Default is 90 and 148 GHz.
- `match` (`bool`): Whether to match the catalogs of detected point sources / clusters to the catalogs of all true point sources / clusters in the maps.
- `match_freqs` (`list` of `int`): Do the matching only at these frequencies. By default (`match_freqs=None`), all frequencies are matched.
- `spectra` (`bool`): Whether to take the power spectra of the maps.
- `spectra_freqs` (`list` of `int`): Calculate the power spectra for these map frequencies. Default is 90 and 148 GHz.
- `plots` (`bool`): Whether to make plots after running all FG cleaning steps. See the notebooks in the `examples` directory to see the set of plots that will be saved.
- `num_mpi_processes` (`int` or `None`): The number of MPI processes you wish to use when FG-cleaning the maps and matching the catalogs. The default (when `num_mpi_processes=None`) is one MPI process per smaller patch in the original, full-sized maps.
- `num_mpi_processes_spectra` (`int`): The number of MPI processes you wish to use when calculating the power spectra of the maps.

Note that if you pass, e.g., `spectra_freqs=[90, 148]` (and same for `mask_freqs`), you will not see those frequencies passed to the `run_hdfgclean.py` file, because those are already the defaults.

In [ ]:
hdfgclean_utils.print_run_hdfgclean_commands(config_file, 
                                             # you may change any of the following:
                                             safe_load=True,
                                             save_maps=True, 
                                             masks=True, 
                                             mask_freqs=fgi.spectra_freqs, 
                                             match=True, 
                                             match_freqs=None, 
                                             spectra=True, 
                                             spectra_freqs=None, 
                                             plots=True,
                                             num_mpi_processes=None,
                                             num_mpi_processes_spectra=2,
                                            )